# Gán Nhãn IATA Delay Code bằng Local LLM (Llama 3.1)
Notebook này thực hiện tiến trình Semantic Data Enrichment bằng cách tích hợp mô hình ngôn ngữ lớn Llama 3 (8B) chạy cục bộ thông qua nền tảng Ollama. Mục tiêu cốt lõi là phân tích các đặc trưng vận hành hỗn hợp (khí tượng, thời gian quay đầu, tải lượng đường băng) để tự động hóa quy trình Xác định nguyên nhân gốc rễ gây trễ chuyến theo danh mục phân loại chuẩn hóa của Hiệp hội Vận tải Hàng không Quốc tế (IATA).

In [ ]:
import os
import json
import time
import pandas as pd
from tqdm import tqdm
from ollama import chat
from pathlib import Path
import numpy as np

### Phân Loại Các Nhãn Delay Code theo IATA
Hệ thống cung cấp cho mô hình một SYSTEM_PROMPT kết hợp kỹ thuật Few-shot Prompting (cung cấp ví dụ mẫu) để ánh xạ các trường dữ liệu số thành quyết định định tính:
* CODE_71 (Weather at Departure): Được kích hoạt khi hệ thống nhận diện Weather_Delay_Risk_Score vượt ngưỡng an toàn (mưa bão, sấm sét tại sân bay đi).
* CODE_93 (Aircraft Rotation): Nguyên nhân trễ dây chuyền, được suy luận trực tiếp khi chỉ số Turnaround_Buffer bị âm (tàu bay về muộn từ chặng trước).
* CODE_89 / CODE_81 (ATC & Airport Congestion): Kích hoạt khi tải lượng sân bay Airport_Load_Factor vượt mức cao điểm (>0.85) hoặc xuất hiện kẹt tắc đường lăn (Taxi_Out_Congestion).
* CODE_06 (No Gate/Stand Available): Tối ưu riêng cho dữ liệu sân bay DAD khi xuất hiện cờ sử dụng bãi đỗ xa Is_Remote_Stand = 1.
* CODE_99: Dành cho các nguyên nhân khác chưa rõ ràng

In [ ]:
SYSTEM_PROMPT = """
Bạn là một chuyên gia điều hành vận hành hàng không (Aviation Operations Expert).
Dựa trên bảng mã IATA Delay Codes chuẩn (AHM730) và các thông số chuyến bay được cung cấp, hãy chẩn đoán nguyên nhân gốc rễ gây trễ chuyến và gán MỘT mã IATA duy nhất, ưu tiên theo thứ tự logic nghiệp vụ sau:

[NHÓM THỜI TIẾT]
- CODE_71 (WO): Thời tiết xấu tại sân bay đi (Weather at departure). Áp dụng nếu Weather_Delay_Risk_Score cao, có rủi ro bão, mưa lớn, hoặc tầm nhìn kém.
- CODE_72 (WT): Thời tiết xấu tại điểm đến (Weather at destination). Áp dụng nếu Destination_Congestion_Risk bị ảnh hưởng bởi thời tiết hoặc có ghi chú.

[NHÓM XOAY VÒNG TÀU BAY - REACTIONARY]
- CODE_93 (RA): Trễ do xoay vòng tàu bay (Aircraft Rotation / Late arrival). Đây là nguyên nhân phổ biến nhất. Áp dụng nếu Turnaround_Buffer < 0 (tàu bay đến trễ), hoặc Accumulated_Delay lớn.

[NHÓM SÂN BAY & KHÔNG LƯU - ATC/AIRPORT]
- CODE_89 (AM): Tắc nghẽn tại sân bay đi (Restrictions at airport of departure). Áp dụng nếu Airport_Load_Factor cao (>0.85) hoặc Flight_Density_Disruption cao.
- CODE_81 (AT): Tắc nghẽn do không lưu (ATFM Restrictions). Áp dụng nếu Taxi_Out_Congestion cao hoặc tắc nghẽn dọc đường bay.
- CODE_06 (OA): Không có sẵn cổng/bãi đỗ (No gate/stand availability). Áp dụng nếu Is_Remote_Stand = 1 (phải dùng xe buýt) kết hợp tải lượng sân bay cao.

[NHÓM KHÁC]
- CODE_99 (MX): Nguyên nhân khác (Other reason). Dùng khi không có tín hiệu rõ ràng từ dữ liệu.

BẮT BUỘC trả về ĐÚNG định dạng JSON như sau, KHÔNG kèm markdown hay bất kỳ văn bản giải thích thừa nào bên ngoài:
{"Delay_Code": "CODE_XX", "Reason": "Giải thích logic dưới 30 chữ dựa trên dữ liệu đầu vào."}
"""

# Cung cấp ví dụ (Few-shot) để đưa vào ngữ cảnh của user
FEW_SHOT_CONTEXT = """
Dưới đây là một số ví dụ phân tích để bạn tham khảo:

Ví dụ 1:
Đầu vào: {"Departure_Delay": 120, "Turnaround_Buffer": -45, "Weather_Delay_Risk_Score": 0, "Airport_Load_Factor": 0.4}
Đầu ra (JSON): {"Delay_Code": "CODE_93", "Reason": "Turnaround Buffer âm 45 phút cho thấy tàu bay đến muộn từ chặng trước, gây trễ dây chuyền."}

Ví dụ 2:
Đầu vào: {"Departure_Delay": 50, "Turnaround_Buffer": 180, "Weather_Delay_Risk_Score": 0.85, "Airport_Load_Factor": 0.3}
Đầu ra (JSON): {"Delay_Code": "CODE_71", "Reason": "Chuyến bay có Weather_Delay_Risk_Score rất cao (0.85), các yếu tố vận hành khác bình thường."}

Ví dụ 3:
Đầu vào: {"Departure_Delay": 40, "Turnaround_Buffer": 60, "Weather_Delay_Risk_Score": 0, "Airport_Load_Factor": 0.96}
Đầu ra (JSON): {"Delay_Code": "CODE_89", "Reason": "Tải lượng sân bay đạt mức 96% gây tắc nghẽn cục bộ tại bãi đỗ và đường lăn."}
"""

def predict_delay_reason_local(flight_data: dict) -> dict:
    user_content = f"{FEW_SHOT_CONTEXT}\n\nPhân tích chuyến bay thực tế sau đây:\n{json.dumps(flight_data, ensure_ascii=False)}"

    max_retries = 3
    for attempt in range(max_retries):
        try:
            if attempt > 0:
                time.sleep(2)  # Giảm thời gian chờ giữa các lần retry một chút

            # Gọi Ollama nhưng cấu hình tối ưu cho Cloud
            response = chat(
                model="ministral-3:3b-cloud",
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': user_content}
                ],
                # 1. BỎ HOÀN TOÀN format='json' để Cloud nhả chữ nhanh nhất có thể
                options={
                    "temperature": 0.0,     # Giữ câu trả lời ổn định, không sáng tạo lung tung
                    "num_predict": 150       # Giới hạn tối đa 150 tokens, tránh model luyên thuyên làm chậm tiến trình
                }
            )

            result_text = response.message.content

            if not result_text or not result_text.strip():
                raise ValueError("Chuỗi trả về từ Cloud rỗng")

            # 2. BỘ LỌC PYTHON: Tự bóc tách chuỗi JSON (Xử lý trường hợp model tự bọc trong ```json ... ```)
            result_text = result_text.strip()
            if "```json" in result_text:
                result_text = result_text.split("```json")[1].split("```")[0].strip()
            elif "```" in result_text:
                result_text = result_text.split("```")[1].split("```")[0].strip()

            # Tìm vị trí ngoặc nhọn đầu và cuối để tránh các đoạn text thừa bên ngoài (nếu có)
            start_idx = result_text.find('{')
            end_idx = result_text.rfind('}')
            if start_idx != -1 and end_idx != -1:
                result_text = result_text[start_idx:end_idx+1]

            return json.loads(result_text)

        except Exception as e:
            if attempt == max_retries - 1:
                # Trả về mã lỗi hợp lệ để không làm sập vòng lặp tqdm lớn ở ngoài
                return {
                    "Delay_Code": "CODE_99",
                    "Reason": f"Lỗi Cloud API sau {max_retries} lần thử. Raw text: {result_text[:50]}... Lỗi: {str(e)}"
                }

### Pipeline Chuẩn MLOps (Checkpoint & Caching)
Để đảm bảo tính khả thi khi xử lý tập dữ liệu lớn trên tài nguyên phần cứng cá nhân, mã nguồn được tích hợp hai cơ chế bảo vệ cốt lõi:
1. Cơ chế Caching (Resume Trạng Thái): Trước khi gọi mô hình, hệ thống tự động đối chiếu các cặp khóa Flight_No và Scheduled_Time với file đầu ra cũ. Những dòng đã được gán nhãn thành công sẽ lập tức được bỏ qua, giúp tiết kiệm tối đa tài nguyên tính toán và thời gian chạy lại hệ thống.
2. Cơ chế Checkpoint (Lưu Trạng Thái Định Kỳ): Script áp dụng một bộ đếm chu kỳ (save_interval = 50). Cứ sau mỗi 50 bản ghi xử lý thành công, dữ liệu tạm thời sẽ được ghi đè xuống đĩa cứng, triệt tiêu hoàn toàn rủi ro mất mát dữ liệu do sự cố phần cứng hoặc ngắt tiến trình đột ngột.

In [ ]:
def main():
    # Khai báo đường dẫn dữ liệu Gold Layer
    current_dir = Path.cwd().parent
    input_path = current_dir / "Data crawl" / "Gold_layer" / "Features" / "master_departure_features_gold.csv"
    output_path = current_dir / "Data crawl" / "Gold_layer" / "Features" / "master_departure_features_gold_annotated.csv"

    if not os.path.exists(input_path):
        print(f"[!] Không tìm thấy file dữ liệu tại {input_path}.")
        return

    print("[*] Đang tải toàn bộ dữ liệu Gold Layer...")
    df = pd.read_csv(input_path)

    # Xây dựng hàm tạo Khóa chính an toàn để quản lý Caching (Bao gồm cả các dòng NaN Scheduled_Time)
    def create_safe_key(row):
        return str(row.get('Flight_No', 'Unknown')) + "_" + str(row.get('Scheduled_Time', 'NO_TIME'))

    # 1. Xử lý Caching / Resume Checkpoint
    flights_to_process = df.copy()
    if os.path.exists(output_path):
        print(f"[*] Tìm thấy file annotated cũ, đang đồng bộ để Resume (chạy tiếp)...")
        df_annotated = pd.read_csv(output_path)

        if 'LLM_Delay_Code' in df_annotated.columns:
            # Lấy danh sách các Khóa chính đã được xử lý (Bao gồm cả dòng đã gán nhãn, dòng Cancelled, dòng Đúng giờ)
            processed_keys = set(df_annotated.apply(create_safe_key, axis=1))

            # Loại bỏ các dòng đã xử lý khỏi danh sách hàng đợi
            flights_to_process = flights_to_process[
                ~flights_to_process.apply(create_safe_key, axis=1).isin(processed_keys)
            ]
            results = df_annotated.to_dict('records')
        else:
            results = []
    else:
        results = []

    print(f"[*] Tổng số chuyến bay cần chạy xử lý: {len(flights_to_process):,}")

    if len(flights_to_process) == 0:
        print("[V] Toàn bộ dữ liệu mạng bay đã được gán nhãn hoàn tất!")
        return

    # 2. Vòng lặp duyệt dữ liệu có Cổng Phân Loại (Logical Gatekeeper)
    save_interval = 50
    count = 0

    for idx, row in tqdm(flights_to_process.iterrows(), total=len(flights_to_process), desc="Tiến trình gán nhãn IATA"):
        result_row = row.to_dict()

        status = str(row.get('Status', '')).upper()
        sched_time = row.get('Scheduled_Time')
        delay = row.get('Departure_Delay', 0)

        # CỔNG 1: Chuyến bay lỗi dữ liệu (Thiếu Giờ dự kiến)
        if pd.isna(sched_time):
            result_row['LLM_Delay_Code'] = np.nan
            result_row['LLM_Delay_Reason'] = 'Lỗi dữ liệu (Thiếu Scheduled_Time)'

        # CỔNG 2: Chuyến bay bị hủy (Cancelled)
        elif 'CANCEL' in status:
            result_row['LLM_Delay_Code'] = np.nan
            result_row['LLM_Delay_Reason'] = 'Chuyến bay bị hủy (Cancelled)'

        # CỔNG 3: Chuyến bay Đúng giờ (Trễ < 15 phút)
        elif pd.isna(delay) or delay < 15:
            result_row['LLM_Delay_Code'] = '-1'
            result_row['LLM_Delay_Reason'] = 'Đúng giờ (chuyến bay trễ dưới 15 phút)'

        # CỔNG 4: Chuyến bay TRỄ (>= 15 phút) -> Kích hoạt LLM
        else:
            flight_payload = {
                "Flight_No": str(row.get('Flight_No', 'Unknown')),
                "Departure_Delay": round(delay, 1),
                "Turnaround_Buffer": round(row.get('Turnaround_Buffer', 0), 1),
                "Accumulated_Delay": round(row.get('Accumulated_Delay', 0), 1),
                "Airport_Load_Factor": round(row.get('Airport_Load_Factor', 0), 2),
                "Destination_Congestion_Risk": round(row.get('Destination_Congestion_Risk', 0), 2),
                "Weather_Delay_Risk_Score": round(row.get('Weather_Delay_Risk_Score', 0), 2),
                "Is_Remote_Stand": 0 if pd.isna(row.get('Is_Remote_Stand')) else int(row.get('Is_Remote_Stand', 0)),
                "Ground_Handling_Pressure": round(row.get('Ground_Handling_Pressure', 0), 2)
            }

            llm_response = predict_delay_reason_local(flight_payload)
            result_row['LLM_Delay_Code'] = llm_response.get('Delay_Code', 'ERROR')
            result_row['LLM_Delay_Reason'] = llm_response.get('Reason', 'Failed to parse JSON')

        # Đẩy kết quả vào danh sách lưu trữ
        results.append(result_row)
        count += 1

        # Kích hoạt lưu trạng thái tự động theo chu kỳ
        if count % save_interval == 0:
            pd.DataFrame(results).to_csv(output_path, index=False, encoding='utf-8-sig')

    # 3. Lưu và xuất bản file cuối cùng
    pd.DataFrame(results).to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n[V] Đã hoàn tất xử lý {len(results):,} dòng! Bảng dữ liệu gán nhãn lưu tại: {output_path}")

if __name__ == "__main__":
    main()